# Skidorternas vintrar – Sälen, Åre, Tärnaby

Kör hela `skiclimate`-pipelinen: hämtar SMHI-stationer, ERA5 och CMIP6 via Open-Meteo, bygger en vintertabell och kör trend-, brytpunkts- och projektionsanalys.

Fungerar i **Google Colab** och i **Microsoft Fabric** (notebook med Python-kernel). Första nedladdningen tar 10–30 minuter beroende på hur Open-Meteo mår. Rådatan sparas under `data/` så steg 2 och 3 kan köras om utan nätverk.

In [ ]:
# Colab: klona repot. Fabric: kör samma sak, eller ladda upp mappen till lakehouse och byt katalog.
import os, sys
if not os.path.exists('AreWeather'):
    !git clone https://github.com/antonalin/AreWeather.git
%cd AreWeather
%pip install -q -r requirements.txt

## 1. Nedladdning

Ta bort `--synthetic` för riktig data. Lägg till `--era5-snow` om du vill ha ERA5:s snödjup (långsamt, ~85 anrop per ort). `--elevation top` beskriver säsongen på toppen istället för i byn.

In [ ]:
!python run_pipeline.py download --resorts all

## 2. Vintertabell och 3. Analys

In [ ]:
!python run_pipeline.py features --resorts all
!python run_pipeline.py analyze

## Titta på resultatet

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

winters = pd.read_parquet('data/processed/winters.parquet')
display(winters.groupby('resort')[['season_days_30cm', 't_mean_winter', 'thaw_days_core']].describe().T)

display(Markdown(open('data/output/summary.md', encoding='utf-8').read()))

In [ ]:
for f in ['trend_season_days_30cm', 'trend_t_mean_winter', 'sensitivity', 'global_coupling', 'projection_season_days_30cm']:
    display(Image(f'data/output/{f}.png'))

## Egen analys

Allt ligger i vanliga DataFrames. `winters` har en rad per ort och vinter; `data/processed/daily.parquet` har dygnsvärdena; `data/output/projection_winters.csv` har varje CMIP6-modells projicerade vintrar.

In [ ]:
from skiclimate import analysis

# exempel: hur känslig är säsongen på toppen istället för i byn?
# (kör features om med --elevation top först)
analysis.season_sensitivity(winters)